In [7]:
%pip install -q numpy pandas spacy
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 11.7 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


# Config

In [8]:
import pandas as pd
import en_core_web_sm

from src.constants import Column, INPUT_DIR, OUTPUT_DIR


# Reusable functions

In [9]:
import re 
import numpy as np

# AI generated Regex pattern to separate sentences
SENTENCE_PATTERN = re.compile(r"[^.!?]+(?:[.!?]+|$)")

def calculate_sentence_count(text: str) -> int:
    sentences = [
        match.group().strip()
        for match in SENTENCE_PATTERN.finditer(str(text))
        if match.group().strip()
    ]

    return len(sentences)

def calculate_character_count(text: str) -> int:
    return sum(character.isalnum() for character in str(text))

def calculate_readability_ari(
    word_count: pd.Series,
    sentence_count: pd.Series,
    character_count: pd.Series,
) -> pd.Series:
    return (
        4.71 * character_count / word_count
        + 0.5 * word_count / sentence_count
        - 21.43
    ).where((word_count > 0) & (sentence_count > 0), np.nan)

def calculate_extremity(rating: float) -> int:
    return int(rating in {1, 5})

In [10]:



nlp = en_core_web_sm.load()

def extract_linguistic_features(doc):
    features = {
        "words": 0,
        "first_person_pronouns": 0,
        "third_person_pronouns": 0,
        "verbs": 0,
        "adjectives": 0,
        "superlatives": 0
    }

    for token in doc:

        # Wortartige Tokens inklusive Zahlen, aber ohne Leerzeichen und Satzzeichen
        if not token.is_space and not token.is_punct:
            features["words"] += 1

        # Personalpronomen nach grammatischer Person
        if token.pos_ == "PRON":
            person = token.morph.get("Person")
            if "1" in person:
                features["first_person_pronouns"] += 1
            elif "3" in person:
                features["third_person_pronouns"] += 1

        # Verben
        if token.pos_ in {"VERB", "AUX"}:
            features["verbs"] += 1

        # Adjektive
        if token.pos_ == "ADJ":
            features["adjectives"] += 1

        # Superlative Adjektive (z. B. best, greatest)
        if token.tag_ in {"JJS", "RBS"}:
            features["superlatives"] += 1

    return features

# Apply Deterministic measured Columns

In [ ]:
def applyDeterministicColumns(input_dir, output_dir):
    input_df = pd.read_csv(input_dir)
    deterministic_df = pd.DataFrame()

    deterministic_df[Column.ID] = input_df[Column.ID]
    deterministic_df[Column.RATING] = input_df[Column.RATING]

    sentence_count = input_df[Column.TEXT].apply(calculate_sentence_count)
    character_count = input_df[Column.TEXT].apply(calculate_character_count)

    documents = nlp.pipe(
        input_df[Column.TEXT].fillna("").astype(str),
        batch_size=64,
    )
    linguistic_features = pd.DataFrame(
        [extract_linguistic_features(doc) for doc in documents],
        index=input_df.index,
    )

    deterministic_df[Column.WORD_COUNT] = linguistic_features["words"]
    deterministic_df[Column.FIRST_PERSON_PRONOUN_COUNT] = linguistic_features["first_person_pronouns"]
    deterministic_df[Column.THIRD_PERSON_PRONOUN_COUNT] = linguistic_features["third_person_pronouns"]
    deterministic_df[Column.VERB_COUNT] = linguistic_features["verbs"]
    deterministic_df[Column.ADJECTIVE_COUNT] = linguistic_features["adjectives"]
    deterministic_df[Column.SUPERLATIVE_COUNT] = linguistic_features["superlatives"]

    deterministic_df[Column.READABILITY_ARI] = calculate_readability_ari(
        deterministic_df[Column.WORD_COUNT],
        sentence_count,
        character_count,
    )

    deterministic_df[Column.EXTREMITY] = input_df[Column.RATING].apply(calculate_extremity)
    
    deterministic_df.to_csv(
        output_dir, 
        index=False, 
        encoding="utf-8"
    )


In [12]:
input_dir = INPUT_DIR / "monitor_reviews_sample_6000.csv"
output_dir = OUTPUT_DIR / "sample_deterministic_features_results.csv"

applyDeterministicColumns(input_dir, output_dir)

KeyError: 'label'